In [1]:
import os
import pandas as pd
import numpy as np
from google.oauth2 import service_account
from google.cloud import bigquery

PROJECT_ID       = "supply-chain-analytics-500607"
DATASET_ID       = "olist"
CREDENTIALS_PATH = "./bq_key.json"

credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

os.makedirs("../data/tableau", exist_ok=True)

def run_query(sql: str) -> pd.DataFrame:
    return client.query(sql).to_dataframe()

def save_csv(df: pd.DataFrame, filename: str) -> None:
    path = f"../data/tableau/{filename}"
    df.to_csv(path, index=False)
    print(f"  ✅ Saved: {filename}  ({df.shape[0]:,} rows × {df.shape[1]} cols)")

TABLE = f"`{PROJECT_ID}.{DATASET_ID}.orders_master`"
print("✅ Config loaded")

✅ Config loaded


In [2]:
# This is your primary Tableau data source
# One row per order with all dimensions and measures

orders_flat = run_query(f"""
    SELECT
        order_id,
        order_status,
        order_purchase_timestamp,
        order_delivered_customer_date,
        order_estimated_delivery_date,
        order_month,
        order_month_num,
        order_year,
        order_dayofweek,
        customer_state,
        customer_city,
        seller_id,
        seller_state,
        seller_city,
        product_category_name_english                       AS category,
        product_weight_g,
        payment_type,
        total_payment_value,
        payment_installments,
        review_score,
        delivery_delay_days,
        shipping_duration_days,
        is_late,
        revenue_at_risk,
        CASE
            WHEN delivery_delay_days < -3  THEN 'Early (3+ days)'
            WHEN delivery_delay_days <= 0  THEN 'On Time'
            WHEN delivery_delay_days <= 3  THEN '1-3 Days Late'
            WHEN delivery_delay_days <= 7  THEN '4-7 Days Late'
            ELSE '8+ Days Late'
        END AS sla_category,
        CASE
            WHEN delivery_delay_days < -3  THEN 1
            WHEN delivery_delay_days <= 0  THEN 2
            WHEN delivery_delay_days <= 3  THEN 3
            WHEN delivery_delay_days <= 7  THEN 4
            ELSE 5
        END AS sla_severity
    FROM {TABLE}
    WHERE order_status != 'canceled'
""")

save_csv(orders_flat, "01_orders_flat.csv")
orders_flat.head(3)

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✅ Saved: 01_orders_flat.csv  (110,189 rows × 26 cols)


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,order_month,order_month_num,order_year,order_dayofweek,customer_state,...,payment_type,total_payment_value,payment_installments,review_score,delivery_delay_days,shipping_duration_days,is_late,revenue_at_risk,sla_category,sla_severity
0,9dc8d1a6f16f1b89874c29c9d8d30447,delivered,2017-10-12 13:33:22,2017-10-24 20:17:44,2017-11-06,2017-10,10,2017,3,MG,...,credit_card,916.02,4.0,5.0,-13.0,12.0,0,0.0,Early (3+ days),1
1,d455a8cb295653b55abda06d434ab492,delivered,2017-09-26 22:17:05,2017-10-07 16:12:47,2017-10-30,2017-09,9,2017,1,PR,...,credit_card,916.02,10.0,5.0,-23.0,10.0,0,0.0,Early (3+ days),1
2,7f39ba4c9052be115350065d07583cac,delivered,2017-10-18 08:16:34,2017-10-27 16:46:05,2017-11-09,2017-10,10,2017,2,MG,...,credit_card,916.02,8.0,1.0,-13.0,9.0,0,0.0,Early (3+ days),1


In [3]:
# Single-row KPI table for Tableau scorecards

kpi = run_query(f"""
    SELECT
        COUNT(DISTINCT order_id)                            AS total_orders,
        ROUND(SUM(is_late) / COUNT(*) * 100, 2)            AS overall_late_pct,
        ROUND(AVG(delivery_delay_days), 2)                  AS avg_delay_days,
        ROUND(SUM(revenue_at_risk), 2)                      AS total_revenue_at_risk,
        ROUND(AVG(review_score), 2)                         AS avg_review_score,
        ROUND(AVG(total_payment_value), 2)                  AS avg_order_value,
        COUNTIF(delivery_delay_days > 7)                    AS severe_late_orders,
        COUNT(DISTINCT seller_id)                           AS total_sellers,
        COUNT(DISTINCT customer_state)                      AS states_covered
    FROM {TABLE}
    WHERE delivery_delay_days IS NOT NULL
""")

save_csv(kpi, "02_kpi_summary.csv")
print(kpi.to_string())

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✅ Saved: 02_kpi_summary.csv  (1 rows × 9 cols)
   total_orders  overall_late_pct  avg_delay_days  total_revenue_at_risk  avg_review_score  avg_order_value  severe_late_orders  total_sellers  states_covered
0         96470              6.59          -12.03             1368508.71              4.05           179.47                3147           2970              27


In [4]:
# For choropleth map and bar charts by state

state_summary = run_query(f"""
    SELECT
        customer_state,
        COUNT(DISTINCT order_id)                            AS total_orders,
        ROUND(SUM(is_late) / COUNT(*) * 100, 2)            AS late_pct,
        ROUND(AVG(delivery_delay_days), 2)                  AS avg_delay_days,
        ROUND(SUM(revenue_at_risk), 2)                      AS revenue_at_risk,
        ROUND(AVG(review_score), 2)                         AS avg_review_score,
        ROUND(AVG(total_payment_value), 2)                  AS avg_order_value,
        COUNTIF(delivery_delay_days > 7)                    AS severe_late_orders
    FROM {TABLE}
    WHERE customer_state IS NOT NULL
      AND delivery_delay_days IS NOT NULL
    GROUP BY customer_state
    ORDER BY late_pct DESC
""")

save_csv(state_summary, "03_state_summary.csv")
print(state_summary.to_string())

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✅ Saved: 03_state_summary.csv  (27 rows × 8 cols)
   customer_state  total_orders  late_pct  avg_delay_days  revenue_at_risk  avg_review_score  avg_order_value  severe_late_orders
0              AL           397     20.84           -8.74         23950.73              3.79           253.92                  43
1              MA           717     18.00           -9.91         40211.20              3.73           243.85                  68
2              SE           335     16.27          -10.00         16452.85              3.88           228.52                  36
3              CE          1279     13.60          -11.10         46038.63              3.86           233.81                 125
4              PI           476     13.58          -11.53         15037.93              3.92           255.54                  29
5              BA          3256     11.89          -10.98         84705.12              3.83           209.93                 209
6              RJ         12350     11

In [5]:
monthly_trend = run_query(f"""
    SELECT
        order_year,
        order_month_num,
        order_month,
        CONCAT(CAST(order_year AS STRING), '-',
               LPAD(CAST(order_month_num AS STRING), 2, '0')) AS year_month,
        COUNT(DISTINCT order_id)                            AS total_orders,
        ROUND(SUM(is_late) / COUNT(*) * 100, 2)            AS late_pct,
        ROUND(AVG(delivery_delay_days), 2)                  AS avg_delay_days,
        ROUND(SUM(revenue_at_risk), 2)                      AS revenue_at_risk,
        ROUND(AVG(review_score), 2)                         AS avg_review_score
    FROM {TABLE}
    WHERE order_month_num IS NOT NULL
      AND delivery_delay_days IS NOT NULL
      AND order_year IS NOT NULL
    GROUP BY order_year, order_month_num, order_month
    HAVING total_orders >= 50
    ORDER BY order_year ASC, order_month_num ASC
""")

save_csv(monthly_trend, "04_monthly_trend.csv")
print(monthly_trend.to_string())

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✅ Saved: 04_monthly_trend.csv  (21 rows × 9 cols)
    order_year  order_month_num order_month year_month  total_orders  late_pct  avg_delay_days  revenue_at_risk  avg_review_score
0         2016               10     2016-10    2016-10           265      0.64          -36.83           198.86              3.85
1         2017                1     2017-01    2017-01           750      2.63          -27.81          3863.20              4.11
2         2017                2     2017-02    2017-02          1653      3.12          -19.18          8657.44              4.16
3         2017                3     2017-03    2017-03          2546      4.69          -12.44         24925.68              4.10
4         2017                4     2017-04    2017-04          2303      6.23          -13.12         34601.15              4.08
5         2017                5     2017-05    2017-05          3545      3.20          -13.39         22743.65              4.17
6         2017                6     20

In [6]:
category_risk = run_query(f"""
    SELECT
        product_category_name_english                       AS category,
        COUNT(DISTINCT order_id)                            AS total_orders,
        ROUND(SUM(is_late) / COUNT(*) * 100, 2)            AS late_pct,
        ROUND(AVG(delivery_delay_days), 2)                  AS avg_delay_days,
        ROUND(SUM(revenue_at_risk), 2)                      AS revenue_at_risk,
        ROUND(AVG(total_payment_value), 2)                  AS avg_order_value,
        ROUND(AVG(review_score), 2)                         AS avg_review_score
    FROM {TABLE}
    WHERE product_category_name_english IS NOT NULL
      AND product_category_name_english != 'unknown'
      AND delivery_delay_days IS NOT NULL
    GROUP BY category
    HAVING total_orders >= 100
    ORDER BY late_pct DESC
""")

save_csv(category_risk, "05_category_risk.csv")
print(f"Categories: {len(category_risk)}")
category_risk.head(10)

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✅ Saved: 05_category_risk.csv  (51 rows × 7 cols)
Categories: 51


,category,total_orders,late_pct,avg_delay_days,revenue_at_risk,avg_order_value,avg_review_score
0,audio,348,11.60,-10.15,7590.27,166.25,3.80
1,christmas_supplies,125,10.00,-12.05,6484.90,125.96,3.88
2,fashion_underwear_beach,117,9.45,-10.93,959.36,97.77,4.02
3,home_confort,392,9.32,-9.81,7010.70,195.09,3.83
4,books_technical,256,7.98,-11.31,1429.40,92.92,4.37
5,office_furniture,1254,7.97,-11.85,47412.35,381.37,3.48
6,baby,2809,7.68,-11.65,44836.43,176.24,4.05
7,electronics,2517,7.59,-11.14,21045.04,90.10,4.04
8,health_beauty,8647,7.56,-11.97,122157.51,171.28,4.16
9,musical_instruments,611,7.37,-11.48,18598.22,339.66,4.18


In [7]:
# Load from BigQuery (already computed in 03_ml_models)
seller_sc = run_query(f"""
    SELECT
        seller_id,
        seller_state,
        total_orders,
        late_rate,
        avg_delay_days,
        avg_review_score,
        total_revenue_at_risk,
        canceled_orders,
        reliability_score,
        reliability_tier
    FROM `{PROJECT_ID}.{DATASET_ID}.seller_scorecard`
    ORDER BY reliability_score ASC
""")

save_csv(seller_sc, "06_seller_scorecard.csv")
print(seller_sc["reliability_tier"].value_counts())

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✅ Saved: 06_seller_scorecard.csv  (1,237 rows × 10 cols)
reliability_tier
Reliable     1235
High Risk       1
Watchlist       1
Name: count, dtype: int64


In [8]:
route_risk = run_query(f"""
    SELECT
        seller_state,
        customer_state,
        route,
        order_count,
        avg_delay_days,
        late_pct,
        revenue_at_risk,
        avg_order_value,
        risk_tier
    FROM `{PROJECT_ID}.{DATASET_ID}.route_risk`
    ORDER BY late_pct DESC
""")

save_csv(route_risk, "07_route_risk.csv")
print(route_risk["risk_tier"].value_counts())

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✅ Saved: 07_route_risk.csv  (100 rows × 9 cols)
risk_tier
Low       76
Medium    21
High       3
Name: count, dtype: int64


In [9]:
sla_analysis = run_query(f"""
    SELECT
        sla_category,
        sla_severity,
        COUNT(order_id)                                     AS order_count,
        ROUND(SUM(total_payment_value), 2)                  AS total_revenue,
        ROUND(AVG(delivery_delay_days), 2)                  AS avg_delay
    FROM `{PROJECT_ID}.{DATASET_ID}.sla_analysis`
    GROUP BY sla_category, sla_severity
    ORDER BY sla_severity ASC
""")

save_csv(sla_analysis, "08_sla_summary.csv")
print(sla_analysis.to_string())

/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✅ Saved: 08_sla_summary.csv  (5 rows × 5 cols)
      sla_category  sla_severity  order_count  total_revenue  avg_delay
0  Early (3+ days)             1        96146    17272455.15     -14.46
1          On Time             2         6779     1133817.66      -1.63
2    1-3 Days Late             3         2110      363056.46       1.83
3    4-7 Days Late             4         2007      392339.34       5.53
4     8+ Days Late             5         3147      613112.91      19.46


In [10]:
import glob

files = sorted(glob.glob("../data/tableau/*.csv"))

print("=" * 60)
print("05_dashboard_prep.ipynb — COMPLETE")
print("=" * 60)
print(f"\n📁 Tableau CSVs saved to ../data/tableau/  ({len(files)} files):")
for f in files:
    df = pd.read_csv(f)
    print(f"   {os.path.basename(f):<35} {df.shape[0]:>7,} rows × {df.shape[1]} cols")

print("""
📌 Next steps in Tableau:
   1. Open Tableau Desktop
   2. Connect → Text File → select 01_orders_flat.csv (primary source)
   3. Add remaining CSVs as separate data sources
   4. Build these sheets:
      • KPI scorecards        (02_kpi_summary.csv)
      • Brazil state map      (03_state_summary.csv)
      • Monthly trend line    (04_monthly_trend.csv)
      • Category risk bars    (05_category_risk.csv)
      • Seller scorecard      (06_seller_scorecard.csv)
      • Route risk table      (07_route_risk.csv)
      • SLA breakdown pie     (08_sla_summary.csv)
   5. Combine into 1 dashboard
   6. Publish to Tableau Public → add link to README
""")

05_dashboard_prep.ipynb — COMPLETE

📁 Tableau CSVs saved to ../data/tableau/  (8 files):
   01_orders_flat.csv                  110,189 rows × 26 cols
   02_kpi_summary.csv                        1 rows × 9 cols
   03_state_summary.csv                     27 rows × 8 cols
   04_monthly_trend.csv                     21 rows × 9 cols
   05_category_risk.csv                     51 rows × 7 cols
   06_seller_scorecard.csv               1,237 rows × 10 cols
   07_route_risk.csv                       100 rows × 9 cols
   08_sla_summary.csv                        5 rows × 5 cols

📌 Next steps in Tableau:
   1. Open Tableau Desktop
   2. Connect → Text File → select 01_orders_flat.csv (primary source)
   3. Add remaining CSVs as separate data sources
   4. Build these sheets:
      • KPI scorecards        (02_kpi_summary.csv)
      • Brazil state map      (03_state_summary.csv)
      • Monthly trend line    (04_monthly_trend.csv)
      • Category risk bars    (05_category_risk.csv)
      • Sel